In [ ]:
!pip install -q torch torchvision diffusers transformers datasets scipy matplotlib

In [ ]:
import math
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import scipy.linalg
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.v2 as T
from datasets import load_dataset
from diffusers import AutoencoderKLFlux2
from torchvision.utils import make_grid
from transformers import AutoModel

warnings.filterwarnings("ignore")

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
VAE_NAME = "black-forest-labs/FLUX.2-klein-4B"
FEATURE_MODEL_NAME = "facebook/dinov2-small"
DATASET_NAME = "huggan/afhq"
CLASS_NAMES = ["cat", "dog", "wild"]
IMAGE_SIZE = 128
IMAGES_PER_CLASS = 1000
TRAIN_STEPS = 500
BATCH_NUM = 64
LEARNING_RATE = 3e-4
# probability of dropping the label so the model also learns the unconditional velocity
LABEL_DROP = 0.1
NUM_EVAL_SAMPLES = 300
SEED = 0

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"DEVICE = {DEVICE}")

# Flow Matching and Diffusion Transformers

Frontier image and video generators (SD3, FLUX.2, Wan, HunyuanVideo) no longer use DDPM with a
U-Net. They train a **transformer** to predict a **velocity** along a straight path from data to
noise. This notebook builds that recipe from its parts (rectified-flow loss, logit-normal
timesteps, adaLN-Zero DiT, Euler/Heun samplers, classifier-free guidance) and trains a small
class-conditional model on real FLUX.2 VAE latents.

---

# 1) Introduction

## 1.1 The problem

Learn a map from Gaussian noise $\epsilon \sim \mathcal{N}(0, I)$ to data $x_0 \sim p_{\text{data}}$
that can be integrated in **few** network evaluations, with a backbone that scales like an LLM.

## 1.2 What was wrong with DDPM + U-Net

| Issue | DDPM / DDIM with a U-Net | Consequence |
|---|---|---|
| Path shape | Variance-preserving schedule $x_t = \sqrt{\bar\alpha_t}x_0 + \sqrt{1-\bar\alpha_t}\epsilon$: a **curved** path | An ODE solver needs many small steps (25–50 even with DDIM) |
| Target | $\epsilon$-prediction is ill-conditioned near $t \to 0$ and $t \to T$ | Schedule tricks (v-prediction, zero-terminal-SNR) needed to fix it |
| Backbone | Convolutional U-Net with attention at some resolutions | Scales poorly; hard to share code and tricks with LLMs; awkward for text tokens |

## 1.3 The core idea

- **Rectified flow**: interpolate linearly, $x_t = (1-t)\,x_0 + t\,\epsilon$, and regress the
  constant velocity $\epsilon - x_0$. Each conditional path is a straight line, and the learned
  marginal flow is close to straight, so few Euler steps go a long way.
- **Diffusion Transformer (DiT)**: patchify the latent into tokens, run a plain transformer, and
  inject the timestep and condition through **adaptive LayerNorm** (adaLN-Zero) instead of
  cross-attention.

## 1.4 Findings from the past year

| Model / method | Date | What is new |
|---|---|---|
| **FLUX.2** (Black Forest Labs) | Nov 2025 | ~32B latent flow-matching model: Mistral-3 24B VLM as the text/condition encoder + rectified-flow transformer; up to 4 MP, multi-reference editing (up to 10 references) |
| **FLUX.2 [klein]** | Jan 2026 | 4B (Apache-2.0) and 9B distilled variants, sub-second generation on consumer GPUs |
| **SD3 / SD3.5** | 2024 | Rectified flow + MM-DiT (separate text and image streams joined in attention); logit-normal $t$ sampling |
| Few-step distillation | 2024–25 | Consistency / shortcut models, adversarial diffusion distillation, distribution-matching distillation: 1–4 steps |
| Representation alignment (REPA) | 2024–25 | Aligning DiT hidden states to a frozen DINO encoder speeds training markedly |

## 1.5 What this notebook covers

| Section | Content |
|---|---|
| 2) Setup | AFHQ (cat / dog / wild) at 128 px, encoded by the FLUX.2 VAE into 32 × 16 × 16 latents |
| 3) The math | Flow ODE, linear path, conditional flow-matching loss, parameterisations, logit-normal $t$, timestep shift, CFG |
| 4) From scratch | Path + loss, adaLN-Zero DiT block and model, Euler/Heun samplers with CFG, Fréchet distance |
| 5) End-to-end | Train a ~10M-parameter DiT for 500 steps (~75 s); samples and FD |
| 6) Variants | Number of steps, Euler vs Heun, guidance scale, timestep shift |
| 7) Practical notes | Scaling, MM-DiT, REPA, distillation, memory |
| 8) Conclusion | Decision table |

---

# 2) Setup

## 2.1 Data and the latent space

Training happens in the latent space of FLUX.2's autoencoder. It downsamples by 8 and has **32
channels**, against 4 for SD 1.x and 16 for SD3/FLUX.1. More channels keep more detail through the
autoencoder, so the generator can work on a small spatial grid. A 128 × 128 image becomes a
$32 \times 16 \times 16$ latent. We use 1,000 images per class from AFHQ and keep the posterior
mean.

In [ ]:
def device_memory_gb() -> float:
    """Memory currently held by the accelerator, in GB."""
    if DEVICE.type == "cuda":
        return torch.cuda.max_memory_allocated() / 1e9
    if DEVICE.type == "mps":
        return torch.mps.driver_allocated_memory() / 1e9
    return 0.0


dataset = load_dataset(DATASET_NAME, split="train")
labels_all = np.array(dataset["label"])
rng = np.random.default_rng(SEED)
subset_idx = np.concatenate(
    [
        rng.choice(np.where(labels_all == c)[0], IMAGES_PER_CLASS, replace=False)
        for c in range(len(CLASS_NAMES))
    ]
)
subset = dataset.select(subset_idx.tolist())
print(
    f"AFHQ train: {len(dataset)} images; using "
    f"{len(subset)} ({IMAGES_PER_CLASS} per class)"
)

vae = AutoencoderKLFlux2.from_pretrained(VAE_NAME, subfolder="vae").to(DEVICE).eval()
print(f"VAE: {sum(p.numel() for p in vae.parameters()) / 1e6:.0f} M params")

In [ ]:
to_tensor = T.Compose(
    [
        T.Resize(IMAGE_SIZE),
        T.CenterCrop(IMAGE_SIZE),
        T.ToImage(),
        T.ToDtype(torch.float32, scale=True),
    ]
)

start = time.time()
latents, pixels = [], []
for i in range(0, len(subset), 64):
    batch = torch.stack(
        [to_tensor(img.convert("RGB")) for img in subset[i : i + 64]["image"]]
    )
    pixels.append((batch * 255).round().to(torch.uint8))
    with torch.no_grad():
        # VAE expects [-1, 1]; (batch_num, 3, 128, 128) -> (batch_num, 32, 16, 16)
        latents.append(vae.encode((batch * 2 - 1).to(DEVICE)).latent_dist.mean.cpu())
latents = torch.cat(latents)
pixels = torch.cat(pixels)
class_labels = torch.tensor(labels_all[subset_idx])
print(
    f"latents {tuple(latents.shape)} | encoded in {time.time() - start:.0f} "
    f"s | device memory {device_memory_gb():.2f} GB"
)

# per-channel standardisation so the data and the noise have comparable scale
latent_mean = latents.mean(dim=(0, 2, 3), keepdim=True)
latent_std = latents.std(dim=(0, 2, 3), keepdim=True)
x0_all = (latents - latent_mean) / latent_std
print(f"raw latent std {latents.std():.2f} -> standardised std {x0_all.std():.2f}")

In [ ]:
@torch.no_grad()
def decode(x0: torch.Tensor) -> torch.Tensor:
    """Standardised latents (batch_num, 32, 16, 16) -> images on CPU.

    Images are (batch_num, 3, 128, 128) in [0, 1].
    """
    images = []
    for i in range(0, len(x0), 50):
        z = x0[i : i + 50].to(DEVICE) * latent_std.to(DEVICE) + latent_mean.to(DEVICE)
        images.append(((vae.decode(z).sample + 1) / 2).clamp(0, 1).cpu())
    return torch.cat(images)


originals = pixels[:8].float() / 255
reconstructed = decode(x0_all[:8])
psnr = -10 * torch.log10(((reconstructed - originals) ** 2).mean())
plt.figure(figsize=(14, 4))
plt.imshow(make_grid(torch.cat([originals, reconstructed]), nrow=8).permute(1, 2, 0))
plt.title(f"top: AFHQ images, bottom: VAE reconstructions (PSNR {psnr:.1f} dB)")
plt.axis("off")
plt.show()

---

# 3) The math

## 3.1 Generating with an ODE

A time-dependent velocity field $v(x, t)$ defines a flow by $\frac{d x_t}{dt} = v(x_t, t)$. If
$x_1 \sim \mathcal{N}(0, I)$ and $v$ is chosen so the density of $x_t$ at $t = 0$ is the data
distribution, then integrating **backwards** from $t=1$ to $t=0$ turns noise into samples.

## 3.2 The linear path and its velocity

Pair each data point $x_0$ with independent noise $\epsilon$ and define

$$
x_t = (1 - t)\, x_0 + t\, \epsilon, \qquad t \in [0, 1].
$$

Along this conditional path the velocity is constant: $\frac{d x_t}{dt} = \epsilon - x_0$.

## 3.3 Conditional flow matching

The velocity that transports the whole distribution is the **marginal** field
$v^\star(x, t) = \mathbb{E}[\epsilon - x_0 \mid x_t = x]$. It is intractable, but regressing the
conditional target has the same minimiser, because the MSE minimiser is the conditional mean:

$$
\mathcal{L}_{\text{CFM}}(\theta) = \mathbb{E}_{t,\, x_0,\, \epsilon}\,
\big\lVert v_\theta(x_t, t, y) - (\epsilon - x_0) \big\rVert^2 .
$$

Here $y$ is the class label. The loss needs no simulation, no noise schedule table, and no
weighting by $\bar\alpha_t$.

## 3.4 One network, three readings

A velocity prediction determines the other usual targets:

$$
\hat x_0 = x_t - t\, v_\theta, \qquad \hat\epsilon = x_t + (1 - t)\, v_\theta .
$$

So $\epsilon$-, $x_0$- and $v$-prediction differ only in how the loss weights each $t$, not in what
can be represented.

## 3.5 Which $t$ to train on

Predicting the velocity is trivial near $t = 0$ (the answer is close to $\epsilon - x_t$) and near
$t = 1$ (close to $x_t - \mathbb{E}[x_0]$). The hard, informative region is the middle. SD3 samples
$t$ from a **logit-normal**

$$
t = \sigma(u), \quad u \sim \mathcal{N}(0, 1),
$$

which concentrates training around $t = 0.5$.

**Timestep shift.** Higher resolution means more pixels share the same signal, so a given $t$ is
effectively less noisy. Samplers compensate by spending more steps near $t = 1$:
$t' = \frac{s\,t}{1 + (s - 1)\,t}$ with shift $s \ge 1$.

## 3.6 Samplers

With $N$ steps on a grid $1 = t_0 > t_1 > \dots > t_N = 0$ and $h_i = t_{i+1} - t_i < 0$:

$$
\text{Euler:}\quad x_{t_{i+1}} = x_{t_i} + h_i\, v_\theta(x_{t_i}, t_i)
$$

$$
\text{Heun:}\quad \tilde x = x_{t_i} + h_i\, v_\theta(x_{t_i}, t_i),\qquad
x_{t_{i+1}} = x_{t_i} + \tfrac{h_i}{2}\big(v_\theta(x_{t_i}, t_i) + v_\theta(\tilde x, t_{i+1})\big).
$$

Euler is exact if paths are straight. Heun's second evaluation corrects for curvature.

## 3.7 Classifier-free guidance

Dropping the label with probability 0.1 during training (replacing it with a null token
$\varnothing$) trains the conditional and unconditional velocity in one network. At sampling time
the two are extrapolated:

$$
\tilde v = v_\theta(x_t, t, \varnothing) + w\,\big(v_\theta(x_t, t, y) - v_\theta(x_t, t, \varnothing)\big),
$$

with $w = 1$ meaning no guidance. Larger $w$ trades diversity for fidelity to the class.

## 3.8 The DiT block with adaLN-Zero

Tokens are $2 \times 2$ latent patches, so $(16/2)^2 = 64$ tokens of dimension $2 \cdot 2 \cdot 32 = 128$,
projected to `embed_dim`. The condition $c = \text{MLP}(\text{emb}(t)) + \text{emb}(y)$ is turned into
per-channel shift, scale and gate parameters for each sub-layer:

$$
(\beta_1, \gamma_1, \alpha_1, \beta_2, \gamma_2, \alpha_2) = W\,\operatorname{SiLU}(c),
$$

$$
h \leftarrow h + \alpha_1 \odot \operatorname{Attn}\big(\operatorname{LN}(h)(1 + \gamma_1) + \beta_1\big),\qquad
h \leftarrow h + \alpha_2 \odot \operatorname{MLP}\big(\operatorname{LN}(h)(1 + \gamma_2) + \beta_2\big).
$$

**Zero** initialisation of $W$ makes every gate $\alpha = 0$, so each block starts as the identity,
the same trick that makes deep residual networks trainable. The DiT paper found this beats
cross-attention or in-context tokens for conditioning.

---

# 4) From-scratch implementation

## 4.1 Path, timestep sampler and loss

In [ ]:
def sample_timesteps(batch_num: int) -> torch.Tensor:
    """Logit-normal t in (0, 1), shape (batch_num,)."""
    return torch.sigmoid(torch.randn(batch_num, device=DEVICE))


def interpolate(x0: torch.Tensor, noise: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    """x_t = (1 - t) x0 + t * noise; t (batch_num,) broadcasts over (C, H, W)."""
    t = t[:, None, None, None]
    return (1 - t) * x0 + t * noise


def flow_matching_loss(
    model: nn.Module, x0: torch.Tensor, labels: torch.Tensor
) -> torch.Tensor:
    """Conditional flow-matching loss ||v_theta(x_t, t, y) - (noise - x0)||^2."""
    noise = torch.randn_like(x0)
    t = sample_timesteps(len(x0))
    pred = model(interpolate(x0, noise, t), t, labels)
    return F.mse_loss(pred.float(), noise - x0)


x0 = x0_all[:4]
noise = torch.randn_like(x0)
print(
    "x_t at t=0 equals x0   : "
    f"{torch.allclose(interpolate(x0, noise, torch.zeros(4)), x0)}"
)
print(
    "x_t at t=1 equals noise: "
    f"{torch.allclose(interpolate(x0, noise, torch.ones(4)), noise)}"
)
t_samples = sample_timesteps(100_000).cpu()
print(
    f"logit-normal t: mean {t_samples.mean():.3f}, "
    "P(0.25 < t < 0.75) = "
    f"{((t_samples > 0.25) & (t_samples < 0.75)).float().mean():.2f} (uniform: 0.50)"
)

## 4.2 Embeddings for time, class and position

Sinusoidal features for $t$ (scaled to $[0, 1000]$, as in DiT), a learned table for the class plus
one extra row for the null label, and fixed 2-D sin-cos position embeddings over the token grid.

In [ ]:
def sincos_1d(dim: int, positions: torch.Tensor) -> torch.Tensor:
    """positions (n,) -> (n, dim) sin/cos features with geometric frequencies."""
    omega = 1.0 / 10000 ** (torch.arange(dim // 2, dtype=torch.float64) / (dim / 2))
    angles = positions.reshape(-1, 1).double() * omega
    return torch.cat([angles.sin(), angles.cos()], dim=1)


def sincos_2d(embed_dim: int, grid_size: int) -> torch.Tensor:
    """(grid_size**2, embed_dim) position embedding: half channels for x, half for y."""
    coords = torch.arange(grid_size, dtype=torch.float64)
    grid_y, grid_x = torch.meshgrid(coords, coords, indexing="ij")
    return torch.cat(
        [sincos_1d(embed_dim // 2, grid_x), sincos_1d(embed_dim // 2, grid_y)], dim=1
    ).float()


class Conditioning(nn.Module):
    """(t, y) -> conditioning vector c of size embed_dim."""

    def __init__(self, num_classes: int, embed_dim: int, freq_dim: int = 256) -> None:
        super().__init__()
        self.freq_dim = freq_dim
        self.time_mlp = nn.Sequential(
            nn.Linear(freq_dim, embed_dim), nn.SiLU(), nn.Linear(embed_dim, embed_dim)
        )
        # last row is the null label used for classifier-free guidance
        self.label_embed = nn.Embedding(num_classes + 1, embed_dim)

    def forward(self, t: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        """t (batch_num,), labels (batch_num,) -> condition (batch_num, embed_dim)."""
        half = self.freq_dim // 2
        freqs = torch.exp(
            -math.log(10000.0) * torch.arange(half, device=t.device) / half
        )
        angles = 1000 * t.float()[:, None] * freqs[None]
        time_feats = torch.cat([angles.cos(), angles.sin()], dim=-1)
        return self.time_mlp(time_feats) + self.label_embed(labels)


pos = sincos_2d(256, 8)
cond = Conditioning(num_classes=3, embed_dim=256)
print(
    f"position embedding {tuple(pos.shape)} | conditioning "
    f"{tuple(cond(torch.rand(4), torch.tensor([0, 1, 2, 3])).shape)}"
)

## 4.3 The adaLN-Zero DiT block

Test: with the zero-initialised modulation, the block must return its input unchanged.

In [ ]:
def modulate(x: torch.Tensor, shift: torch.Tensor, scale: torch.Tensor) -> torch.Tensor:
    """x (batch_num, num_tokens, embed_dim); shift, scale (batch_num, embed_dim)."""
    return x * (1 + scale[:, None]) + shift[:, None]


class DiTBlock(nn.Module):
    """Pre-LN transformer block; the condition sets its norms and residual gates."""

    def __init__(self, embed_dim: int, num_heads: int, mlp_ratio: int = 4) -> None:
        super().__init__()
        self.num_heads = num_heads
        self.norm1 = nn.LayerNorm(embed_dim, elementwise_affine=False, eps=1e-6)
        self.qkv = nn.Linear(embed_dim, 3 * embed_dim)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim, elementwise_affine=False, eps=1e-6)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_ratio * embed_dim),
            nn.GELU(approximate="tanh"),
            nn.Linear(mlp_ratio * embed_dim, embed_dim),
        )
        self.modulation = nn.Linear(embed_dim, 6 * embed_dim)
        # adaLN-Zero: all gates start at 0, so the block starts as the identity
        nn.init.zeros_(self.modulation.weight)
        nn.init.zeros_(self.modulation.bias)

    def forward(self, x: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
        """x (batch_num, num_tokens, embed_dim), c (batch_num, embed_dim) -> like x."""
        batch_num, num_tokens, embed_dim = x.shape
        shift1, scale1, gate1, shift2, scale2, gate2 = self.modulation(F.silu(c)).chunk(
            6, dim=-1
        )
        h = modulate(self.norm1(x), shift1, scale1)
        # (batch_num, num_tokens, 3 * embed_dim)
        # -> 3 x (batch_num, num_heads, num_tokens, head_dim)
        q, k, v = (
            self.qkv(h)
            .view(batch_num, num_tokens, 3, self.num_heads, -1)
            .permute(2, 0, 3, 1, 4)
        )
        attn = (
            F.scaled_dot_product_attention(q, k, v)
            .transpose(1, 2)
            .reshape(batch_num, num_tokens, embed_dim)
        )
        x = x + gate1[:, None] * self.proj(attn)
        return x + gate2[:, None] * self.mlp(modulate(self.norm2(x), shift2, scale2))


block = DiTBlock(embed_dim=256, num_heads=4)
x_tokens = torch.randn(2, 64, 256)
c = cond(torch.rand(2), torch.tensor([0, 1]))
print(
    f"block output {tuple(block(x_tokens, c).shape)}; identity "
    f"at init: {torch.equal(block(x_tokens, c), x_tokens)}"
)

## 4.4 The full DiT

Patchify with a strided convolution, add position embeddings, run the blocks, then apply a final
modulated LayerNorm and a zero-initialised linear layer that maps each token back to a
$2 \times 2 \times 32$ patch. At initialisation the model predicts $v = 0$ everywhere.

```
x_t (B, 32, 16, 16) ─ patchify 2×2 ─► (B, 64, embed_dim) + pos
                                          │  num_layers × DiTBlock(·, c)
                                          ▼
                       adaLN + Linear ─► (B, 64, 2·2·32) ─ unpatchify ─► v (B, 32, 16, 16)
```

In [ ]:
class DiT(nn.Module):
    """Class-conditional diffusion transformer that predicts a latent velocity field."""

    def __init__(
        self,
        in_channels: int = 32,
        input_size: int = 16,
        patch_size: int = 2,
        embed_dim: int = 256,
        num_layers: int = 8,
        num_heads: int = 4,
        num_classes: int = 3,
    ) -> None:
        super().__init__()
        self.patch_size = patch_size
        self.in_channels = in_channels
        self.patchify = nn.Conv2d(in_channels, embed_dim, patch_size, patch_size)
        self.register_buffer(
            "pos",
            sincos_2d(embed_dim, input_size // patch_size)[None],
            persistent=False,
        )
        self.cond = Conditioning(num_classes, embed_dim)
        self.blocks = nn.ModuleList(
            [DiTBlock(embed_dim, num_heads) for _ in range(num_layers)]
        )
        self.norm_out = nn.LayerNorm(embed_dim, elementwise_affine=False, eps=1e-6)
        self.modulation_out = nn.Linear(embed_dim, 2 * embed_dim)
        self.out = nn.Linear(embed_dim, patch_size * patch_size * in_channels)
        for layer in [self.modulation_out, self.out]:
            nn.init.zeros_(layer.weight)
            nn.init.zeros_(layer.bias)
        nn.init.normal_(self.cond.label_embed.weight, std=0.02)

    def forward(
        self, x: torch.Tensor, t: torch.Tensor, labels: torch.Tensor
    ) -> torch.Tensor:
        """x (batch_num, C, H, W), t (batch_num,), labels (batch_num,).

        Returns the velocity (batch_num, C, H, W).
        """
        batch_num, channels, height, width = x.shape
        # (batch_num, C, H, W) -> (batch_num, num_tokens, embed_dim)
        h = self.patchify(x).flatten(2).transpose(1, 2) + self.pos
        c = self.cond(t, labels)
        for block in self.blocks:
            h = block(h, c)
        shift, scale = self.modulation_out(F.silu(c)).chunk(2, dim=-1)
        h = self.out(modulate(self.norm_out(h), shift, scale))
        p, grid_h, grid_w = (
            self.patch_size,
            height // self.patch_size,
            width // self.patch_size,
        )
        # (batch_num, num_tokens, p * p * C) -> (batch_num, C, H, W)
        h = h.view(batch_num, grid_h, grid_w, p, p, channels).permute(0, 5, 1, 3, 2, 4)
        return h.reshape(batch_num, channels, height, width)

At initialisation the output is exactly zero, so the loss equals $\mathbb{E}\lVert \epsilon - x_0
\rVert^2$ per element, which is 2 for unit-variance data.

In [ ]:
model = DiT(num_classes=len(CLASS_NAMES)).to(DEVICE)
out = model(
    x0_all[:4].to(DEVICE), torch.rand(4, device=DEVICE), class_labels[:4].to(DEVICE)
)
print(
    f"DiT: {sum(p.numel() for p in model.parameters()) / 1e6:.1f} M params | "
    f"output {tuple(out.shape)} | all zero at init: {bool((out == 0).all())}"
)
initial_loss = flow_matching_loss(
    model, x0_all[:256].to(DEVICE), class_labels[:256].to(DEVICE)
)
print(
    f"initial loss {initial_loss:.3f} "
    "(expected 2.0: E||noise - x0||^2 per element with unit-variance data)"
)

## 4.5 Samplers with guidance and timestep shift

In [ ]:
@torch.no_grad()
def sample(
    model: nn.Module,
    labels: torch.Tensor,
    num_steps: int,
    method: str = "euler",
    guidance: float = 1.0,
    shift: float = 1.0,
    seed: int = SEED,
) -> torch.Tensor:
    """Euler/Heun from t=1 (noise) to t=0 -> latents (batch_num, 32, 16, 16)."""
    generator = torch.Generator().manual_seed(seed)
    x = torch.randn(len(labels), 32, 16, 16, generator=generator).to(DEVICE)
    ts = torch.linspace(1, 0, num_steps + 1, device=DEVICE)
    ts = shift * ts / (1 + (shift - 1) * ts)
    null = torch.full_like(labels, len(CLASS_NAMES))

    def velocity(x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """Guided velocity at x (batch_num, 32, 16, 16) and scalar time t."""
        t = t.expand(len(x))
        if guidance == 1.0:
            return model(x, t, labels)
        v_cond, v_uncond = model(
            torch.cat([x, x]), torch.cat([t, t]), torch.cat([labels, null])
        ).chunk(2)
        return v_uncond + guidance * (v_cond - v_uncond)

    for t, t_next in zip(ts[:-1], ts[1:]):
        v = velocity(x, t)
        if method == "euler":
            x = x + (t_next - t) * v
        else:
            x_pred = x + (t_next - t) * v
            x = x + (t_next - t) * (v + velocity(x_pred, t_next)) / 2
    return x


labels_demo = torch.arange(3, device=DEVICE)
print(
    f"untrained model, 4 Euler steps -> {tuple(sample(model, labels_demo, 4).shape)}; "
    f"a zero velocity leaves pure noise, std {sample(model, labels_demo, 4).std():.2f}"
)

## 4.6 Fréchet distance in DINOv2 feature space

FID compares the Gaussian statistics of Inception features of real and generated images. Recent
work prefers **DINOv2** features (FD$_\text{DINOv2}$), which track human judgement better:

$$
\operatorname{FD} = \lVert \mu_r - \mu_g \rVert^2 + \operatorname{tr}\big(\Sigma_r + \Sigma_g - 2(\Sigma_r \Sigma_g)^{1/2}\big).
$$

With a few hundred samples the absolute value is biased upward, so it is used here **only to
compare settings**. The real-versus-real distance at the same sample size is the floor.

In [ ]:
feature_model = AutoModel.from_pretrained(FEATURE_MODEL_NAME).to(DEVICE).eval()
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)


@torch.no_grad()
def dino_features(images: torch.Tensor) -> np.ndarray:
    """images (n, 3, H, W) in [0, 1] -> (n, 384) DINOv2 pooled features."""
    feats = []
    for i in range(0, len(images), 100):
        x = F.interpolate(
            images[i : i + 100].to(DEVICE), size=224, mode="bicubic", antialias=True
        ).clamp(0, 1)
        feats.append(
            feature_model(
                pixel_values=(x - IMAGENET_MEAN) / IMAGENET_STD
            ).pooler_output.cpu()
        )
    return torch.cat(feats).double().numpy()


def frechet_distance(feats_a: np.ndarray, feats_b: np.ndarray) -> float:
    """Fréchet distance between Gaussians fitted to two (n, d) feature sets."""
    mu_a, mu_b = feats_a.mean(0), feats_b.mean(0)
    cov_a, cov_b = np.cov(feats_a, rowvar=False), np.cov(feats_b, rowvar=False)
    cov_sqrt = scipy.linalg.sqrtm(cov_a @ cov_b).real
    return float(((mu_a - mu_b) ** 2).sum() + np.trace(cov_a + cov_b - 2 * cov_sqrt))


perm = torch.randperm(len(pixels), generator=torch.Generator().manual_seed(SEED))
real_feats = dino_features(pixels[perm[:1000]].float() / 255)
floor_feats = dino_features(pixels[perm[1000 : 1000 + NUM_EVAL_SAMPLES]].float() / 255)
print(
    f"FD(real {NUM_EVAL_SAMPLES} vs real 1000) = "
    f"{frechet_distance(real_feats, floor_feats):.1f}  <- floor"
)

---

# 5) End-to-end: training a DiT with flow matching

500 steps of batch 64 on 3,000 latents (about 11 epochs), AdamW, bf16 autocast. That is enough to
learn fur texture, colours and the rough face layout of each class, but not crisp eyes, ears or
outlines. Production runs
train for hundreds of thousands of steps at batch 256–4096.

In [ ]:
torch.manual_seed(SEED)
model = DiT(num_classes=len(CLASS_NAMES)).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.0)
x0_device, labels_device = x0_all.to(DEVICE), class_labels.to(DEVICE)

losses = []
start = time.time()
for step in range(TRAIN_STEPS):
    idx = torch.randint(0, len(x0_device), (BATCH_NUM,), device=DEVICE)
    labels = labels_device[idx].clone()
    labels[torch.rand(BATCH_NUM, device=DEVICE) < LABEL_DROP] = len(CLASS_NAMES)
    with torch.autocast(DEVICE.type, dtype=torch.bfloat16):
        loss = flow_matching_loss(model, x0_device[idx], labels)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if step % 100 == 0 or step == TRAIN_STEPS - 1:
        print(f"step {step:4d} | loss {loss.item():.3f} | {time.time() - start:5.0f} s")
print(
    f"training: {time.time() - start:.0f} s | device memory {device_memory_gb():.2f} GB"
)

plt.figure(figsize=(7, 3))
plt.plot(np.convolve(losses, np.ones(20) / 20, mode="valid"))
plt.xlabel("step")
plt.ylabel("flow-matching loss (20-step mean)")
plt.show()

In [ ]:
model.eval()
grid_labels = torch.arange(3, device=DEVICE).repeat_interleave(8)
samples = decode(sample(model, grid_labels, num_steps=16, guidance=3.0))
plt.figure(figsize=(14, 6))
plt.imshow(make_grid(samples, nrow=8).permute(1, 2, 0))
plt.title("16 Euler steps, guidance 3 — rows: cat, dog, wild")
plt.axis("off")
plt.show()

---

# 6) Variants

## 6.1 Steps, solver, guidance and shift

The same trained model, sampled differently and scored by FD. Number of network evaluations
(NFE) counts guidance as 2 per step.

In [ ]:
eval_labels = torch.arange(NUM_EVAL_SAMPLES, device=DEVICE) % len(CLASS_NAMES)
settings = [
    ("euler", 1, 1.0, 1.0),
    ("euler", 4, 1.0, 1.0),
    ("euler", 16, 1.0, 1.0),
    ("heun", 8, 1.0, 1.0),
    ("euler", 16, 3.0, 1.0),
    ("euler", 16, 3.0, 3.0),
]
print("solver | steps | guidance | shift | NFE | FD     | seconds")
for method, num_steps, guidance, shift in settings:
    start = time.time()
    images = decode(sample(model, eval_labels, num_steps, method, guidance, shift))
    fd = frechet_distance(real_feats, dino_features(images))
    nfe = num_steps * (2 if method == "heun" else 1) * (2 if guidance != 1.0 else 1)
    print(
        f"{method:6s} | {num_steps:5d} | {guidance:8.1f} | {shift:5.1f} "
        f"| {nfe:3d} | {fd:6.1f} | {time.time() - start:5.1f}"
    )

---

# 7) Practical notes

**Scaling up.** The production recipe changes sizes, not the mechanism: more parameters
(SD3.5-medium 2.5B, FLUX.2 [klein] 4B, FLUX.2 32B), higher resolution with **resolution
bucketing** (batches of equal aspect ratio), and text conditioning. For text, SD3 introduced
**MM-DiT**: text and image tokens keep separate weights (their own QKV and MLP) but attend
jointly in one attention over the concatenated sequence. FLUX.2 replaces CLIP/T5 with a 24B VLM as
the condition encoder.

**Shift grows with resolution.** At 1024 px, SD3 and FLUX use a shift around 3, and FLUX makes it
depend on the token count. At 128 px the effect is small.

**Speeding up training.** REPA adds a loss aligning an early DiT layer's tokens with a frozen DINOv2
encoder's patch features. It reports convergence up to ~17× faster for the same FID. It costs
one frozen encoder forward per batch.

**Speeding up sampling.** Straight paths make few steps possible, but quality at 1–4 steps needs
distillation: consistency/shortcut models (learn to jump along the ODE), adversarial diffusion
distillation (a discriminator on the few-step output), or distribution-matching distillation.
FLUX.2 [klein] is a distilled few-step model.

**Memory.** Activation memory grows with tokens squared through attention. A 1024 px image with an
8× VAE and 2 × 2 patches is 4,096 tokens, so production training uses FlashAttention, gradient
checkpointing, and bf16 with an EMA of the weights for sampling.

**Failure modes at demo scale.** Short training gives the right colours and layout but soft
textures and malformed details (eyes, ears). FD keeps improving long after these 500 steps.

---

# 8) Conclusion

| Choice | DDPM-era default | Flow-matching-era default | Why it changed |
|---|---|---|---|
| Path | Variance-preserving, curved | Linear interpolation (rectified flow) | Straighter trajectories, fewer steps |
| Target | $\epsilon$ | Velocity $\epsilon - x_0$ | Well-conditioned at both ends of $t$ |
| $t$ sampling | Uniform | Logit-normal | Train where the task is hardest |
| Backbone | U-Net | DiT / MM-DiT with adaLN-Zero | Scales like an LLM; text and image share attention |
| Latent | 4-channel VAE | 16–32-channel VAE | More detail through the autoencoder |
| Sampler | 50-step DDIM | 20–50 Euler/Heun; 1–4 after distillation | ODE is nearly straight; distillation closes the rest |